In [0]:
# Cell 1 — Create richer dataset
from pyspark.sql.functions import col, avg, rank, desc
from pyspark.sql.window import Window

data = [
    ("Alice", "ML Engineer", 92, "TeamA"),
    ("Bob", "Data Engineer", 85, "TeamB"),
    ("Carol", "ML Engineer", 95, "TeamA"),
    ("Dave", "Data Engineer", 78, "TeamB"),
    ("Eve", "ML Engineer", 88, "TeamA"),
    ("Frank", "Data Scientist", 91, "TeamC")
]
df = spark.createDataFrame(
    data, ["name", "role", "score", "team"]
)

In [0]:
# Cell 2 — Transformations (lazy — nothing runs yet!)
filtered = df.filter(col("score") > 85)
grouped = df.groupBy("role").agg(
    avg("score").alias("avg_score")
)

In [0]:
# Cell 3 — Actions (these trigger computation!)
filtered.show()
grouped.orderBy(desc("avg_score")).show()

+-----+--------------+-----+-----+
| name|          role|score| team|
+-----+--------------+-----+-----+
|Alice|   ML Engineer|   92|TeamA|
|Carol|   ML Engineer|   95|TeamA|
|  Eve|   ML Engineer|   88|TeamA|
|Frank|Data Scientist|   91|TeamC|
+-----+--------------+-----+-----+

+--------------+-----------------+
|          role|        avg_score|
+--------------+-----------------+
|   ML Engineer|91.66666666666667|
|Data Scientist|             91.0|
| Data Engineer|             81.5|
+--------------+-----------------+



In [0]:
# Cell 4 — Window function in Spark!
window = Window.partitionBy("role").orderBy(
    desc("score")
)
from pyspark.sql.functions import row_number
df.withColumn("rank", row_number().over(window)).show()


+-----+--------------+-----+-----+----+
| name|          role|score| team|rank|
+-----+--------------+-----+-----+----+
|  Bob| Data Engineer|   85|TeamB|   1|
| Dave| Data Engineer|   78|TeamB|   2|
|Frank|Data Scientist|   91|TeamC|   1|
|Carol|   ML Engineer|   95|TeamA|   1|
|Alice|   ML Engineer|   92|TeamA|   2|
|  Eve|   ML Engineer|   88|TeamA|   3|
+-----+--------------+-----+-----+----+



In [0]:
# Cell 5 — Check execution plan
filtered.explain()


== Physical Plan ==
LocalTableScan [name#13264, role#13265, score#13266L, team#13267]


== Photon Explanation ==
Photon does not fully support the query because:
		Unsupported node: LocalTableScan [name#13264, role#13265, score#13266L, team#13267].

Reference node:
	LocalTableScan [name#13264, role#13265, score#13266L, team#13267]

